# Surgical Vision — reproduce all experiments on Colab

1. Runtime → Change runtime type → **GPU** (A100 recommended) with high RAM.
2. Put the course data in Google Drive under `MyDrive/BN5211/` (`Dataset/`, `dataset-full/`, `RARP_1FPS/`).
3. Results are written to `MyDrive/BN5211/runs/`. Finished runs are skipped, so after a disconnect run the setup cells again and re-run the interrupted project cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d repo && git -C repo pull --ff-only || git clone https://github.com/weirdquantum/surgical-vision-moco.git repo
%cd /content/repo
!pip install -q pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/BN5211'
CACHE = f'{DRIVE}/cache/data_cache.tar'   # Dataset + dataset-full + resized RARP frames (~140 MB)
!mkdir -p /content/data {DRIVE}/runs {DRIVE}/cache
!ln -sfn /content/data data
!ln -sfn {DRIVE}/runs runs
import os
if os.path.exists(CACHE):
    # Later sessions: one large file copies in seconds (Drive is slow per file).
    !tar -xf {CACHE} -C /content/data
else:
    # First session: copy thousands of files from Drive (5-15 min), resize, then cache.
    !rsync -a --info=progress2 {DRIVE}/Dataset {DRIVE}/dataset-full {DRIVE}/RARP_1FPS /content/data/
    !python -m action_recognition.prepare --data-root data --workers 8
    !tar -cf {CACHE} -C /content/data Dataset dataset-full RARP_1FPS_288x512
!ls data && du -sh data/*

In [ ]:
!python -m pytest -q tests

## Project 1 — instrument classification (~1.5 h on an A100)

In [ ]:
!bash scripts/reproduce.sh p1 2>&1 | tee -a runs/colab_p1.log

## Project 2 — action recognition (~1.5 h)

In [ ]:
!bash scripts/reproduce.sh p2 2>&1 | tee -a runs/colab_p2.log

## Project 3 — MoCo pretraining and fine-tuning (~3.5 h)

In [ ]:
!bash scripts/reproduce.sh p3 2>&1 | tee -a runs/colab_p3.log

## All results

In [ ]:
!python scripts/collect_results.py --runs runs --output runs/ALL_RESULTS.md > /dev/null
!cat runs/ALL_RESULTS.md